# Agentic Retrieval-Augmented Generation (RAG) System

This notebook builds a document question-answering system using a RAG architecture.

Pipeline:

PDF → Chunking → Embeddings → Vector Database → Semantic Retrieval → LLM Answer

Technologies used:

- LangChain
- Sentence Transformers (all-MiniLM-L6-v2)
- Qdrant Vector Database
- Llama 3.1 via Groq
- Gradio Web Interface

# Imports

In [1]:
import os
from pypdf import PdfReader
from dotenv import load_dotenv
from langchain.schema import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.tools import tool
from langchain.agents import create_react_agent, AgentExecutor
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain_groq import ChatGroq
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import gradio as gr

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print("✅ API Loaded:", GROQ_API_KEY is not None)

✅ API Loaded: True


In [2]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
client = QdrantClient(":memory:")
collection_name = "rag_collection"
llm = ChatGroq(model="llama-3.1-8b-instant", api_key=GROQ_API_KEY, temperature=0)
print("Models ready")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models ready


In [3]:
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

def search_pdf(question):
    query_embedding = embedding_model.embed_query(question)
    results = client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=12
    )
    return "\n\n".join([r.payload["text"] for r in results.points])

def ask(question):
    search_query = question + " education skills projects experience achievements"
    context = search_pdf(search_query)
    history = memory.load_memory_variables({})["chat_history"]
    history_text = ""
    for msg in history:
        role = "User" if msg.type == "human" else "Assistant"
        history_text += f"{role}: {msg.content}\n"
    prompt = (
        "You are an intelligent assistant analyzing a document.\n"
        "Answer the question based on the context provided below.\n"
        "Be detailed, helpful and conversational.\n\n"
        f"Previous conversation:\n{history_text}\n\n"
        f"Context from document:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )
    response = llm.invoke(prompt)
    answer = response.content
    memory.save_context({"input": question}, {"output": answer})
    return answer

def analyze(jd_text):
    if not indexed:
        return "Please upload a resume first.", "", ""

    context = search_pdf("skills experience projects education achievements")

    # Better match scoring — breakdown by category
    match_prompt = (
        "You are an expert technical recruiter.\n"
        "Analyze the resume against the job description and give a detailed match score.\n\n"
        "Score these 4 categories out of 25 each:\n"
        "1. Technical Skills Match (does resume have the required tech stack?)\n"
        "2. Experience Level Match (years/depth of experience vs what JD needs?)\n"
        "3. Project Relevance (are the projects relevant to this role?)\n"
        "4. Soft Skills & Other Requirements\n\n"
        "Format your response exactly like this:\n"
        "Technical Skills: XX/25 - [one line reason]\n"
        "Experience Level: XX/25 - [one line reason]\n"
        "Project Relevance: XX/25 - [one line reason]\n"
        "Soft Skills: XX/25 - [one line reason]\n"
        "TOTAL: XX/100\n"
        "Summary: [2 lines overall assessment]\n\n"
        f"Resume:\n{context}\n\n"
        f"Job Description:\n{jd_text}\n\n"
        "Breakdown:"
    )

    # Gap analysis with multiple searches for better coverage
    skills_context = search_pdf("technical skills programming languages frameworks tools")
    exp_context = search_pdf("work experience internship projects")

    gap_prompt = (
        "You are an expert technical recruiter.\n"
        "Compare this resume carefully to the job description.\n"
        "List the TOP 5 specific gaps — skills, experience, or qualifications\n"
        "that are clearly required in the JD but missing or weak in the resume.\n"
        "For each gap, suggest HOW the candidate can address it.\n\n"
        "Format exactly like this:\n"
        "GAP 1: [missing skill/experience]\n"
        "HOW TO FIX: [specific suggestion]\n\n"
        "GAP 2: [missing skill/experience]\n"
        "HOW TO FIX: [specific suggestion]\n\n"
        "(continue for all 5)\n\n"
        f"Resume Skills:\n{skills_context}\n\n"
        f"Resume Experience:\n{exp_context}\n\n"
        f"Job Description:\n{jd_text}\n\n"
        "Gaps:"
    )

    # Interview questions personalized to BOTH resume and JD
    interview_prompt = (
        "You are an expert technical interviewer.\n"
        "Generate 7 interview questions that are SPECIFICALLY tailored to this candidate\n"
        "applying for this role. Reference their actual projects and skills.\n\n"
        "Mix:\n"
        "- 3 technical questions based on the JD requirements\n"
        "- 2 project-specific questions based on their resume\n"
        "- 2 behavioral questions relevant to the role\n\n"
        "Number them 1-7. For project questions, mention the actual project name.\n\n"
        f"Candidate Resume:\n{context}\n\n"
        f"Job Description:\n{jd_text}\n\n"
        "Personalized Interview Questions:"
    )

    match_result = llm.invoke(match_prompt).content
    gap_result = llm.invoke(gap_prompt).content
    interview_result = llm.invoke(interview_prompt).content
    return match_result, gap_result, interview_result

print("Ready")

Ready


In [4]:
indexed = False

def index_pdf(pdf_file):
    global indexed
    reader = PdfReader(pdf_file)
    text = ""
    for page in reader.pages:
        text += page.extract_text()

    if not text.strip():
        return "Could not extract text from PDF. Please try a different file."

    documents = [Document(page_content=text)]
    chunks = text_splitter.split_documents(documents)
    embeddings = embedding_model.embed_documents(
        [chunk.page_content for chunk in chunks]
    )
    if client.collection_exists(collection_name):
        client.delete_collection(collection_name)
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=768, distance=Distance.COSINE)
    )
    points = [
        PointStruct(id=i, vector=emb, payload={"text": chunks[i].page_content})
        for i, emb in enumerate(embeddings)
    ]
    client.upsert(collection_name=collection_name, points=points)
    indexed = True
    return "Resume indexed! You can now analyze it."

def chat(message, history):
    if not indexed:
        return "Please upload a resume first."
    return ask(message)

with gr.Blocks(title="Resume Analyzer", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Resume Analyzer with LLaMA 3.1")
    gr.Markdown("Upload your resume and paste a job description to get a detailed match score, gap analysis and personalized interview questions.")

    with gr.Row():
        pdf_input = gr.File(label="Upload Resume (PDF)", file_types=[".pdf"])
        status = gr.Textbox(label="Status", interactive=False)

    pdf_input.change(fn=index_pdf, inputs=pdf_input, outputs=status)

    gr.Markdown("## Job Description Analyzer")
    jd_input = gr.Textbox(
        label="Paste Job Description here",
        lines=8,
        placeholder="Paste the job description you want to apply for..."
    )
    analyze_btn = gr.Button("Analyze Resume", variant="primary", size="lg")

    # Loading state
    with gr.Row(visible=False) as loading_row:
        gr.Markdown("### Analyzing your resume... please wait")

    with gr.Row():
        match_output = gr.Textbox(label="Match Score Breakdown", lines=10)
        gap_output = gr.Textbox(label="Gap Analysis + How to Fix", lines=10)

    interview_output = gr.Textbox(label="Personalized Interview Questions", lines=12)

    analyze_btn.click(
        fn=lambda: gr.update(visible=True),
        outputs=loading_row
    ).then(
        fn=analyze,
        inputs=[jd_input],
        outputs=[match_output, gap_output, interview_output]
    ).then(
        fn=lambda: gr.update(visible=False),
        outputs=loading_row
    )

    gr.Markdown("## Chat with your Resume")
    gr.ChatInterface(fn=chat)

demo.launch()

/tmp/ipykernel_19511/3606520580.py:37: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Resume Analyzer", theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
